In [4]:
from pathlib import Path
import geopandas as gpd

# Build a list of residential addresses in Heerlen from the Limburg OSM PBF.
pbf_path = Path("../limburg-260312.osm.pbf")
output_path = Path("../output/heerlen_residential_addresses.csv")

# Read only OSM points. Address house numbers are commonly stored as points in OSM.
points = gpd.read_file(pbf_path, layer="points")


def get_tag_series(frame: gpd.GeoDataFrame, key: str):
    """Return a string series for an OSM tag from a direct column or from other_tags."""
    if key in frame.columns:
        return frame[key].fillna("").astype(str).str.strip()

    if "other_tags" in frame.columns:
        escaped_key = key.replace(":", r"\:")
        pattern = rf'"{escaped_key}"=>"([^"]*)"'
        return frame["other_tags"].fillna("").astype(str).str.extract(pattern, expand=False).fillna("").str.strip()

    return frame.index.to_series().map(lambda _: "")


street_series = get_tag_series(points, "addr:street")
housenumber_series = get_tag_series(points, "addr:housenumber")
postcode_series = get_tag_series(points, "addr:postcode")
city_series = get_tag_series(points, "addr:city")
building_series = get_tag_series(points, "building")

# Keep only records that look like postal addresses.
address_mask = street_series.ne("") | housenumber_series.ne("")
addresses = points.loc[address_mask].copy()

# Materialize normalized address fields so downstream steps are schema-agnostic.
addresses["addr:street"] = street_series.loc[address_mask].values
addresses["addr:housenumber"] = housenumber_series.loc[address_mask].values
addresses["addr:postcode"] = postcode_series.loc[address_mask].values
addresses["addr:city"] = city_series.loc[address_mask].values
addresses["building"] = building_series.loc[address_mask].values

# Restrict to Heerlen based on city tag.
city_mask = addresses["addr:city"].str.lower().str.strip() == "heerlen"
addresses = addresses.loc[city_mask].copy()

# Restrict to residential addresses when a building type is known.
building_normalized = addresses["building"].str.lower().str.strip()
has_building_value = building_normalized.ne("")
residential_value = building_normalized.isin(["residential", "house"])
addresses = addresses.loc[~has_building_value | residential_value].copy()

# Add lon/lat columns from geometry for map and nearest-neighbor workflows.
addresses = addresses.to_crs(4326)
addresses["lon"] = addresses.geometry.x
addresses["lat"] = addresses.geometry.y

# Select useful columns for output.
result = addresses[
    [
        "addr:street",
        "addr:housenumber",
        "addr:postcode",
        "addr:city",
        "building",
        "lon",
        "lat",
    ]
].copy()

# Remove duplicates and rows without the minimum address fields.
result = result.dropna(subset=["addr:street", "addr:housenumber"])
result = result[(result["addr:street"].str.strip() != "") & (result["addr:housenumber"].str.strip() != "")]
result = result.drop_duplicates().reset_index(drop=True)

output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False)

print(f"Loaded {len(points)} points with columns: {', '.join(points.columns)}")
print(f"Saved {len(result)} Heerlen residential addresses to {output_path}")


Loaded 794272 points with columns: osm_id, name, barrier, highway, ref, address, is_in, place, man_made, other_tags, geometry
Saved 43409 Heerlen residential addresses to ..\output\heerlen_residential_addresses.csv
